In [ ]:
%pip install --upgrade --quiet  langchain langchain-openai langchain-experimental presidio-analyzer presidio-anonymizer spacy Faker

In [ ]:
text = """
Discharge summary,Report,,,"Admission Date:  [**2118-6-2**]       Discharge Date:  [**2118-6-14**]

Date of Birth:                    Sex:  F

Service:  MICU and then to [**Doctor Last Name **] Medicine
Attending:name Luis
HISTORY OF PRESENT ILLNESS:  This is an 81-year-old female
with a history of emphysema (not on home O2), who presents
with three days of shortness of breath thought by her primary
care doctor to be a COPD flare.  Two days prior to admission,
she was started on a prednisone taper and one day prior to
admission she required oxygen at home in order to maintain
oxygen saturation greater than 90%.  She has also been on
levofloxacin and nebulizers, and was not getting better, and
presented to the [**Hospital1 18**] Emergency Room.

In the [**Hospital3 **] Emergency Room, her oxygen saturation was
100% on CPAP.  She was not able to be weaned off of this
despite nebulizer treatment and Solu-Medrol 125 mg IV x2.

Review of systems is negative for the following:  Fevers,
chills, nausea, vomiting, night sweats, change in weight,
gastrointestinal complaints, neurologic changes, rashes,
palpitations, orthopnea.  Is positive for the following:
Chest pressure occasionally with shortness of breath with
exertion, some shortness of breath that is positionally
related, but is improved with nebulizer treatment.

PAST MEDICAL HISTORY:
1. COPD.  Last pulmonary function tests in [**2117-11-3**]
demonstrated a FVC of 52% of predicted, a FEV1 of 54% of
predicted, a MMF of 23% of predicted, and a FEV1:FVC ratio of
67% of predicted, that does not improve with bronchodilator
treatment.  The FVC, however, does significantly improve with
bronchodilator treatment consistent with her known reversible
air flow obstruction in addition to an underlying restrictive
ventilatory defect.  The patient has never been on home
oxygen prior to this recent episode.  She has never been on
steroid taper or been intubated in the past.
2. Lacunar CVA.  MRI of the head in [**2114-11-4**]
demonstrates



"""

In [ ]:
text, service, doctor = extract_and_redact_service_attending(text)

print(service, doctor)

In [26]:
import csv
from tqdm import tqdm
from itertools import islice
import re

def extract_and_redact_service_attending(text):
    """
    Extracts and redacts Service and Attending from a hospital note.
    Returns: (redacted_text, service, doctor)
    """
    service_match = re.search(r"Service:\s*(.+)", text)
    attending_match = re.search(r"Attending:\s*name\s+(.+)", text)

    service = service_match.group(1).strip() if service_match else None
    doctor = attending_match.group(1).strip() if attending_match else None

    # Redact from text
    if service_match:
        text = re.sub(r"Service:\s*.+", "Service: [REDACTED]", text)
    if attending_match:
        text = re.sub(r"Attending:\s*name\s+.+", "Attending: [REDACTED]", text)

    return text, service, doctor

In [ ]:
import re
from tqdm import tqdm
import csv

def extract_and_redact_service_attending(text):
    service_match = re.search(r"Service:\s*(.+)", text)
    attending_match = re.search(r"Attending:\s*name\s+(.+)", text)

    service = service_match.group(1).strip() if service_match else None
    doctor = attending_match.group(1).strip() if attending_match else None

    if service_match:
        text = re.sub(r"Service:\s*.+", "Service: [REDACTED]", text)
    if attending_match:
        text = re.sub(r"Attending:\s*name\s+.+", "Attending: [REDACTED]", text)

    return text, service, doctor

def process_file(input_file, service_doctor_csv, redacted_output_csv):
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    # Split entries by delimiter "
    entries = content.split('"')

    with open(service_doctor_csv, 'w', encoding='utf-8', newline='') as sdOutfile, \
         open(redacted_output_csv, 'w', encoding='utf-8', newline='') as redactedOutfile:

        sdWriter = csv.writer(sdOutfile)
        redactedWriter = csv.writer(redactedOutfile)

        sdWriter.writerow(['service', 'attending'])
        redactedWriter.writerow(['entry'])  # single column CSV with redacted note

        for entry in tqdm(entries):
            entry = entry.strip()
            if not entry:
                continue
            redacted_text, service, doctor = extract_and_redact_service_attending(entry)

            if service and doctor:
                sdWriter.writerow([service, doctor])

            redactedWriter.writerow([redacted_text])

input_csv_path = "NOTEEVENTS_deano.csv"
service_doctor_csv_path = "service_doctor_pairs.csv"
redacted_output_csv_path = "NOTEEVENTS_REMOVED_PAIRS.csv"

process_file(input_csv_path, service_doctor_csv_path, redacted_output_csv_path)


In [ ]:
read_phrasePool("phrase_context_pool.csv", 5)

In [ ]:
import csv
count = 0
with open('phrase_context_pool', newline='', encoding='ISO-8859-1') as csvfile:
          tweet_csv = csv.reader(csvfile)
          for row in tweet_csv:
                count+=1
                

In [ ]:
count

In [32]:
with open("NOTEEVENTS_deanonymized.csv", 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line[:100]}")
        if i > 10:
            break


Line 0: TEXT

Line 1: "Admission Date:  date 2023-06-04       Discharge Date:  date 2023-11-22

Line 2: 

Line 3: Date of Birth:                    Sex:  F

Line 4: 

Line 5: Service:  MICU and then to name Brown Medicine

Line 6: 

Line 7: HISTORY OF PRESENT ILLNESS:  This is an 81-year-old female

Line 8: with a history of emphysema (not on home O2), who presents

Line 9: with three days of shortness of breath thought by her primary

Line 10: care doctor to be a COPD flare.  Two days prior to admission,

Line 11: she was started on a prednisone taper and one day prior to

